In [1]:
from importlib import reload
import soundata
from hierarchies.hierarchies import get_hierarchy_tree
from writers import write_csv
from pathlib import Path
import pandas as pd
import ast

In [2]:
DATASET = "urbansound8k" # Choose any available dataset from the 'soundata' library

In [3]:
dataset = soundata.initialize(DATASET)
try:
    print("Validating if Dataset files exists:")
    dataset.validate()
except:
    print("Dataset could not be validated, downloading:")
    dataset.download()  # download the dataset
    dataset.validate()  # validate that all the expected files are there

HIERARCHY_TREE = get_hierarchy_tree(DATASET)
HIERARCHY_TREE.print_tree()

Validating if Dataset files exists:


100%|██████████| 8732/8732 [00:07<00:00, 1130.16it/s]
INFO: Success: the dataset is complete and all files are valid.
INFO: --------------------


root
    - human_animal [level=0, idx=0]
        - human [level=1, idx=0]
            - children_playing [level=2, idx=2]
        - animal [level=1, idx=1]
            - dog_bark [level=2, idx=3]
    - vehicle [level=0, idx=1]
        - vehicle_operation [level=1, idx=2]
            - car_horn [level=2, idx=1]
            - engine_idling [level=2, idx=5]
        - vehicle_signal [level=1, idx=3]
            - siren [level=2, idx=8]
    - mechanical [level=0, idx=2]
        - construction [level=1, idx=4]
            - drilling [level=2, idx=4]
            - jackhammer [level=2, idx=7]
        - ventilation [level=1, idx=5]
            - air_conditioner [level=2, idx=0]
        - signal [level=1, idx=6]
            - gun_shot [level=2, idx=6]
    - music [level=0, idx=3]
        - recorded [level=1, idx=7]
            - street_music [level=2, idx=9]


In [4]:
HIERARCHY_TREE.get_path(leaf_idx=2, output="names")

['human_animal', 'human', 'children_playing']

In [25]:
# Save MetaDataset to CSV

# Output folder
out_dir = Path("metadata")
out_dir.mkdir(parents=True, exist_ok=True)
output_path = out_dir / "{}_metadata.csv".format(DATASET)

rows = []
for clip_id in dataset.clip_ids:
    clip = dataset.clip(clip_id)

    hierarchy_indices = HIERARCHY_TREE.get_path(leaf_idx=clip.class_id, output="indices")

    rows.append({
        "clip_id": clip.clip_id,
        "class_id": clip.class_id,
        "freesound_start_time": clip.freesound_start_time,
        "freesound_end_time": clip.freesound_end_time,
        "salience": clip.salience,
        "slice_file_name": clip.slice_file_name,
        
        "class_label": clip.class_label,
        "hierarchy": hierarchy_indices
    })
write_csv(output_path, rows)

Saved 8732 clips to: metadata\urbansound8k_metadata.csv


In [26]:
# Save Dataset to CSV
# We will save a designated dataframe (CSV) coressponding to a distinct dataset in data/.
# Therefore it will only be necessary to run the Data Adapter once pr. dataset and the constructed dataframe
# can just be directly passed straight into the AudioDataset henceforth.

out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)


meta_path = Path(f"metadata/{DATASET}_metadata.csv")
df_meta = pd.read_csv(meta_path)

rows = []

for _, row in df_meta.iterrows():

    clip_id = row["clip_id"]

    # Get clip from soundata
    clip = dataset.clip(clip_id)

    # Build audio path
    audio_path = clip.audio_path  # already correct path

    # Parse hierarchy (stored as string in CSV)
    hierarchy = row["hierarchy"]
    if isinstance(hierarchy, str):
        hierarchy = ast.literal_eval(hierarchy)

    rows.append({
        "clip_id": clip_id,
        "audio_path": str(audio_path),
        "class_id": int(row["class_id"]),
        "class_label": row["class_label"],
        "fold": int(clip.fold),
        "hierarchy": hierarchy
    })

df_out = pd.DataFrame(rows)

# Save compiled dataset
output_path = out_dir / f"{DATASET}_compiled.csv"
df_out.to_csv(output_path, index=False)

print(f"Saved {len(df_out)} samples to: {output_path}")

Saved 8732 samples to: data\urbansound8k_compiled.csv


In [5]:
import random
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from pathlib import Path
import soundfile as sf

In [6]:
from model_frameworks.model_utilities import ConvBlock
from model_frameworks.dataloader_utilities import AudioDataset, AudioTransform

In [9]:
df = pd.read_csv("data/urbansound8k_compiled.csv")

# Check file path validation
if not Path(df.iloc[0]["audio_path"]).exists():
    print("File path for audio data does NOT exist.")
    print("- If running on a new device, try re-running the data adapter...")
else:
    print("File path for audio exists.")

transform = AudioTransform(sample_rate=22050, n_mels=64) # remember to init the log-Mel spectrogram transformer
dataset = AudioDataset(df=df, transform=transform, split="train")
sample = dataset[0]

print("Input shape:", sample["input"].shape)
print("Target:", sample["target"])
print("Hierarchy:", sample["meta"]["hierarchy"])
print("Index:", sample["index"])
print("Meta:", sample["meta"])
print("target_level_0:", sample["target_level_0"])
print("target_level_1:", sample["target_level_1"])
print("target_level_2:", sample["target_level_2"])

File path for audio exists.
Input shape: torch.Size([1, 64, 442])
Target: tensor(2)
Hierarchy: [0, 0, 2]
Index: 0
Meta: {'clip_id': '135776-2-0-49', 'label_name': 'children_playing', 'hierarchy': '[0, 0, 2]', 'fold': 1}
target_level_0: tensor(0)
target_level_1: tensor(0)
target_level_2: tensor(2)
